# Independent device analysis - completed solution

Select **PySpark**, attach the facilitator-assigned default Lakehouse, and
confirm pre-delivery readiness before running. No Lakehouse/workspace IDs,
credentials, outputs, or preselected environment are embedded.

Worked answers are supplied; explain them rather than treating a green run as learner completion.

Source: `10_independent_challenge.md and artifacts/build_notebooks.py (challenge_cells)` in the supplied training package. Relative documentation links
work in the repository/package context and may not resolve after Fabric import;
keep the matching handout and notebook import guide open separately.

# Independent challenge: Evidence, not just a green notebook

After the successful main notebook, answer the question for D032 on October 10
versus October 6, 2026 UTC. Use existing accepted Silver for metrics and source
exports/quarantine for quality evidence. No writes, source edits, installs, or
permission changes. Setup supplies context only, not a transformation recipe.
See `10_independent_challenge.md` in the supplied training package for the rubric.

In [ ]:
from pyspark.sql import functions as F

spark.conf.set("spark.sql.session.timeZone", "UTC")
DEVICE_ID = 'D032'
BASELINE_DATE, TARGET_DATE = ('2026-10-06', '2026-10-10')
BATCH_ID = "s04-lab-devices-001"
SCHEDULED_READINGS = 288
for table in ("s04_lab_device_silver", "s04_lab_device_bronze", "s04_lab_device_rejected"):
    if not spark.catalog.tableExists(table):
        raise RuntimeError("Run the successful main notebook in this default Lakehouse first.")
challenge_events = spark.table("s04_lab_device_silver").filter(F.col("batch_id") == BATCH_ID)
challenge_bronze = spark.table("s04_lab_device_bronze").filter(F.col("batch_id") == BATCH_ID)
challenge_rejected = spark.table("s04_lab_device_rejected").filter(F.col("batch_id") == BATCH_ID)

## Task 1: Summarize the two days

Return device/day/site/type, accepted count, min/max/mean temperature, scheduled
count, accepted-slot shortfall, and accepted percentage. Use accepted events,
two UTC dates, and a clearly stated grain. Do not copy a pre-aggregated answer.

In [ ]:
challenge_summary = (
    challenge_events.filter(F.col("device_id") == DEVICE_ID)
    .withColumn("reading_date", F.to_date("event_time_utc"))
    .filter(F.col("reading_date").isin(BASELINE_DATE, TARGET_DATE))
    .groupBy("reading_date", "device_id", "site_id", "device_type")
    .agg(
        F.count("*").alias("reading_count"),
        F.min("temperature_c").alias("min_temperature_c"),
        F.max("temperature_c").alias("max_temperature_c"),
        F.avg("temperature_c").cast("decimal(8,2)").alias("avg_temperature_c"),
    )
    .withColumn("scheduled_readings", F.lit(SCHEDULED_READINGS))
    .withColumn("accepted_slot_shortfall", SCHEDULED_READINGS - F.col("reading_count"))
    .withColumn("accepted_percent", F.round(F.col("reading_count") / SCHEDULED_READINGS * 100, 2))
)
display(challenge_summary.orderBy("reading_date"))

## Task 2: Explain the source populations

Count raw source rows and quarantine reasons for each requested daily export.
Include a zero-rejection day explicitly. Use source_file for export scope:
an invalid timestamp cannot establish that record's true event date.

In [ ]:
def export_scope(df):
    return (
        df.filter(F.upper(F.trim("device_id")) == DEVICE_ID)
        .withColumn("source_export_date", F.regexp_extract(
            "source_file", r"readings_([0-9]{4}-[0-9]{2}-[0-9]{2})\.csv$", 1
        ))
        .filter(F.col("source_export_date").isin(BASELINE_DATE, TARGET_DATE))
    )


source_counts = export_scope(challenge_bronze).groupBy("source_export_date").agg(
    F.count("*").alias("raw_source_rows")
)
rejection_counts = export_scope(challenge_rejected).groupBy(
    "source_export_date", "quality_issue"
).count()
rejection_totals = rejection_counts.groupBy("source_export_date").agg(
    F.sum("count").alias("quarantine_rows")
)
quality_summary = source_counts.join(
    rejection_totals, "source_export_date", "left"
).fillna({"quarantine_rows": 0})
display(quality_summary.orderBy("source_export_date"))
display(rejection_counts.orderBy("source_export_date", "quality_issue"))

## Presenter answer key

| UTC date | Accepted | Min C | Max C | Mean C | Shortfall / 288 | Accepted % |
| --- | --- | --- | --- | --- | --- | --- |
| 2026-10-06 | 256 | 2.00 | 9.30 | 5.45 | 32 | 88.89 |
| 2026-10-10 | 287 | 2.00 | 9.40 | 5.44 | 1 | 99.65 |

D032 is a COLD_STORE device at S04. The change between reported rounded daily means is **-0.01 C** (target minus baseline).

October 6 has 291 raw rows and 35 quarantine rows: 10 DUPLICATE_EVENT, 10 INVALID_TEMPERATURE, 5 INVALID_TIMESTAMP, 5 MISSING_EVENT_KEY, and 5 OUT_OF_RANGE. October 10 has 287 raw rows and **zero** quarantine rows. Filling a missing aggregate count with zero for reporting is not filling invalid sensor values with zero.

October 10 still lacks one accepted observation against the known 288-slot schedule. Zero quarantine does not prove complete ingestion or device uptime. The baseline has 32 slots without accepted readings; raw/quarantine counts include duplicate occurrences and are different populations. Invalid-time records are scoped by source export, not assigned repaired event timestamps.

A -0.01 C difference between rounded means is not evidence of a trend, statistical significance, device health, or a firmware-related cause. Do not average those rounded means into an overall event mean. Compute ran on Fabric Spark over persisted Lakehouse tables; this exercise writes nothing and proves no Warehouse/model freshness.

Restart your own session and rerun to demonstrate independence from other notebooks' variables. Evaluate the learner's explanation, not just whether this completed solution ran.